### Imports

In [ ]:
from qarp.blocks import TrotterBlock
from qarp.operators import JordanWigner
from qarp.operators.models import fermi_hubbard
from qarp.algorithms import DOSQPE, find_occupation_numbers, find_eigenspectrum_degeneracy
from qarp.algorithms import SpectrumEstimator

from qarp.operators.functions import eigenspectrum
from qarp.operators import FullyCommuting
import numpy as np

### DOS-QPE object

In [ ]:
# Hamiltonian Trotterization

# DOS-QPE parameters
n_qubits = 4
n_ancilla = 6

# Trotterization parameters
time = 2 * np.pi
n_trotter_steps = 4
trotter_order = 2
grouping = FullyCommuting()

# choose between probing the full spectrum (n_hamming = None) or only a specific occupation number (e.g. n_hamming = 2)
n_hamming = None

fham = fermi_hubbard((2,), t=0.14, U=0.231)
qham = JordanWigner().encode_operator(fham)
eigs = eigenspectrum(qham)
if eigs[0] < 0:
    qham = qham + np.abs(eigs[0]) + np.finfo(float).eps # manual shift to keep the spectrum between 0 and 1

# n=1
# qham = 0.3 + QubitOperator("Z0", 0.2) 
# qham = 0.55 + QubitOperator("Z0", 0.2) 
# n=2
# qham = 0.45 + QubitOperator("Z0 Z1", 0.2) + QubitOperator("X0 Y1", 0.2) 
# n=3
# qham = 0.45 + QubitOperator("Z0 Z1 Z2", 0.2) + QubitOperator("X0 Y1 Y2", 0.1) + QubitOperator("Y0 X1 X2", 0.2) - QubitOperator("X0 Y1", 0.05) - QubitOperator("Y0 X1", 0.05)
# qham = 0.5 + QubitOperator("Z0 Z1 Z2", 0.2) + QubitOperator("X0 Y1 Y2", 0.1) + QubitOperator("Y0 X1 X2", 0.2) - QubitOperator("X0 Y1", 0.05) - QubitOperator("Y0 X1", 0.05)
# qham = QubitOperator("Z0 Z1", 0.2) + QubitOperator("X0 Y1", 0.3) - QubitOperator("Y0 X1", 0.4)

eigs = eigenspectrum(qham)
print("Eigenvalues: ", eigs) # these will be the eigenvalues we want to estimate through the DOSQPE algorithm

# create the Trotter circuit generated by the Hamiltonian.
# Note the minus sign on the time: TrotterBlock builds U = exp(-iH*time), so we
# pass -time to get U = exp(+iH*time). DOS-QPE then reads the phases phi = E directly.
# (With +time the phases come out mirrored as phi = 1 - E.)
circ_unitary = TrotterBlock(operator=qham, n_qubits=n_qubits, steps=n_trotter_steps, time=-time, order=trotter_order, grouping=grouping).build()

# create the DOSQPE object
dosqpe = DOSQPE(circ_unitary, n_ancilla, n_hamming).build()

### DOS-QPE circuits

In [ ]:
# access the DOS-QPE state preparation circuit
dosqpe.state.plot()

In [ ]:
# access the DOS-QPE full circuit
dosqpe.unitary.plot(scrollable=True)

### Run the algorithm

In [ ]:
# run the DOS-QPE sampling and build the distribution
dosqpe.run()

### Plots

In [ ]:
# plot the distribution
dosqpe.plot()

### Plot against real spectrum

In [ ]:
# evaluate the occupation numbers
occ_numbers = find_occupation_numbers(qham, n_qubits)

In [ ]:
# generate a dictionary of the eigenvalues and their degeneracy
degeneracy_dict = find_eigenspectrum_degeneracy(eigs)
for ele in degeneracy_dict.items():
    print("Eigenvalue: ", ele[0], "Degeneracy: ", ele[1])
# create a list of the unique eigenvalues
unique_eigs = list(degeneracy_dict.keys())
# create a list of the degeneracies normalized to 1
normalized_degeneracy = [degeneracy_dict[key] / sum(degeneracy_dict.values()) for key in unique_eigs]

In [ ]:
# find the unique occupation numbers based on the unique eigenvalues
# NB: eig_tolerance must be set to the same value as the tolerance used in the find_degenerate_eigenvalues function - default is 1e-15
eig_tolerance = 1e-15

unique_occ_numbers = np.zeros(len(unique_eigs))
for idx, eig in enumerate(unique_eigs):
    index = np.where(np.isclose(eigs, eig, atol=eig_tolerance))[0][0] # find the index of the eigenvalue in the eigs array
    unique_occ_numbers[idx] = occ_numbers[index]
unique_occ_numbers = np.round(unique_occ_numbers).astype(int) # convert to int
print(unique_occ_numbers)

In [ ]:
# plot the distribution and the real spectrum
dosqpe.plot_against_spectrum(unique_eigs, normalized_degeneracy, unique_occ_numbers) #, lw=5)

### Spectrum reconstruction

In [ ]:
distribution = dosqpe.distribution

estimator = SpectrumEstimator(
    n_qubits=n_qubits,
    n_ancilla=n_ancilla,
    mode='auto',
    peak_guess="find_peaks",  # use 'find peaks' (default) or 'uniform'
    verbose=True
)

phases, degeneracies = estimator.estimate(
    distribution=dosqpe.distribution,
    freqs=dosqpe.freqs,
    n_candidates=100,
    l1_penalty=1,
    cluster=True,
    cluster_method='gap', # use 'fixed' or 'gap'
    cluster_tolerance=1.0 / (2 ** (n_ancilla + 1)), # only used if cluster_method='fixed'
    cluster_gap_threshold=0.5, # only used if cluster_method='gap'
    threshold=0.2
)

estimator.plot(
    distribution=dosqpe.distribution, 
    freqs=dosqpe.freqs,
    true_phases=unique_eigs, 
    true_degeneracies=normalized_degeneracy,
    show_deconvolved=True
)

### Dicke state probe: fixed number of particles

In [ ]:

n_hamming = 2  # two particles, i.e. half-filled 2-site Fermi-Hubbard model
dosqpe = DOSQPE(circ_unitary, n_ancilla, n_hamming).build()
_ = dosqpe.run()

In [ ]:
dosqpe.plot_against_spectrum(unique_eigs, normalized_degeneracy, unique_occ_numbers)

In [ ]:
estimator = SpectrumEstimator(
    n_qubits=n_qubits,
    n_ancilla=n_ancilla,
    mode='l2',
    peak_guess="find_peaks",
    hamming_weight=2,
    verbose=True
)

phases, degeneracies = estimator.estimate(
    distribution=dosqpe.distribution,
    freqs=dosqpe.freqs,
    l1_penalty=1,
    cluster=True,
    cluster_method='gap', # use 'fixed' or 'gap'
    cluster_tolerance=1.0 / (2 ** (n_ancilla + 1)), # only used if cluster_method='fixed'
    cluster_gap_threshold=0.5, # only used if cluster_method='gap'
    threshold=0.2
)

estimator.plot(
    distribution=dosqpe.distribution, 
    freqs=dosqpe.freqs,
    show_deconvolved=True
)